In [ ]:
import torch
import sys
import os
src_path = os.path.abspath(os.path.join('..', 'src'))
if src_path not in sys.path:
    sys.path.append(src_path)
from semantic_topic_extension import EmbeddingMapper
from model import (
    load_model, 
    load_sentence_model, 
    detect,
)
from watermark.auto_watermark import AutoWatermark
from watermark.transformers_config import TransformersConfig
from datasets import load_dataset
import numpy as np
import nltk
nltk.download('punkt_tab')

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

args = {
    # 'model_name_or_path': 'facebook/opt-2.7b',
    # 'model_name_or_path': 'facebook/opt-1.3b',
#     'model_name_or_path': 'facebook/opt-125m', 
    # 'model_name_or_path': 'meta-llama/Llama-3.2-1B',
#     'model_name_or_path': 'facebook/opt-6.7b',
    'model_name_or_path': 'google/gemma-7b',
    'load_fp16' : True,
    'prompt_max_length': None, 
    'max_new_tokens': 200, 
    'generation_seed': 123, 
    'use_sampling': True, 
    'n_beams': 1, 
    'sampling_temp': 0.7, 
    'use_gpu': True, 
    'seeding_scheme': 'simple_1', 
    'delta': 2.0, 
    'ignore_repeated_bigrams': False, 
    'detection_z_threshold': 4.75, 
    'select_green_tokens': True,
    'skip_model_load': False,
    'seed_separately': True,
    'topic_token_mapping': {},
    'granularity': 'kmeans',
}

In [ ]:
topic_list = ["animals", "technology", "sports", "medicine"]

model, tokenizer = load_model(args)
print("Model loaded")

In [ ]:
c4_dataset = load_dataset('json', data_files='./c4/c4.json', split='train[:1000]')
c4_iter = iter(c4_dataset)
inputs_list = []
n = 1000  # number of samples to create
for _ in range(n):
    entry = next(c4_iter)
    text = entry["text"]
    token_ids = tokenizer.encode(text, add_special_tokens=False)
    prompt_ids = token_ids[:200]
    prompt = tokenizer.decode(prompt_ids, clean_up_tokenization_spaces=True)
    inputs_list.append(prompt)

In [ ]:
sentence_model = load_sentence_model()
print("Sentence model loaded")
embedding_mapper = EmbeddingMapper(tokenizer, sentence_model)
total_tokens, vocab_embeddings = embedding_mapper.get_model_vocab_embeddings()
topic_embeddings = embedding_mapper.get_defined_topic_list_embeddings(topic_list)
topic_token_mapping = embedding_mapper.map_tokens_to_topics(total_tokens, vocab_embeddings, topic_list, topic_embeddings, threshold=0.9)
args['topic_token_mapping'] = topic_token_mapping

In [ ]:
transformers_config = TransformersConfig(
    model=model,
    tokenizer=tokenizer,
#     vocab_size=50272,
    vocab_size=256000,
    device=device,
    max_new_tokens=200+5,
    min_new_tokens=200-5,
    do_sample=True,
    no_repeat_ngram_size=4
)

In [ ]:
def detect_KGW(watermarked_text, transformers_config):
    myWatermark = AutoWatermark.load('KGW', transformers_config=transformers_config)
    detect_result_watermarked = myWatermark.detect_watermark(watermarked_text)
    return detect_result_watermarked

def detect_DIP(watermarked_text, transformers_config):
    myWatermark = AutoWatermark.load('DIP', transformers_config=transformers_config)
    detect_result_watermarked = myWatermark.detect_watermark(watermarked_text)
    return detect_result_watermarked

def detect_Unigram(watermarked_text, transformers_config):
    myWatermark = AutoWatermark.load('Unigram', transformers_config=transformers_config)
    detect_result_watermarked = myWatermark.detect_watermark(watermarked_text)
    return detect_result_watermarked

def detect_SynthID(watermarked_text, transformers_config):
    myWatermark = AutoWatermark.load('SynthID', transformers_config=transformers_config)
    detect_result_watermarked = myWatermark.detect_watermark(watermarked_text)
    return detect_result_watermarked

In [ ]:
schemes_detection = {
    "TBW": detect,
#     "KGW": detect_KGW,
#     "DIP": detect_DIP,
#     "Unigram": detect_Unigram,
#     "SynthID": detect_SynthID,
}

detected_results = {}
for scheme in schemes_detection:
    print(scheme)
    detection_function = schemes_detection.get(scheme)
    detection_results = []
    if scheme == 'TBW':
        for human_text in inputs_list:
            result_formatted = detection_function(human_text, human_text, args, device=device, tokenizer=tokenizer, sentence_model=sentence_model)
            result = {
                'is_watermarked': result_formatted[6][1] == 'Watermarked',
                'score': float(result_formatted[3][1])
            }
            print(result)
            detection_results.append(result)
    else:
        for human_text in inputs_list:
            result = detection_function(human_text, transformers_config)
            print(result)
            detection_results.append(result)
    detected_results[scheme] = detection_results

In [ ]:
for scheme, results in detected_results.items():
    total_results = len(results)
    true_count = sum(1 for result in results if result['is_watermarked'])
    true_rate = (true_count / total_results) * 100 if total_results else None
    scores = [result['score'] for result in results]
    
    mean_score = np.mean(scores) if scores else None
    std_score = np.std(scores) if scores else None
    
    print(f"{scheme}:")
    print(f"  Total true 'is_watermarked': {true_count}")
    print(f"  True positive rate: {true_rate:.2f}%")
    print(f"  Mean score: {mean_score:.3f}")
    print(f"  Standard Deviation: {std_score:.3f}\n")